[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/exorbyte/mbox-cookbook/blob/main/1-getting-started/01_installation_and_quickstart.ipynb)
# Installation & Quickstart

Welcome to the **M|BOX** cookbook 👋

M|BOX is an identity resolution and fuzzy matching engine built for Python and Pandas workflows. It resolves messy, real-world records, typos, nicknames, accented characters, formatting differences, into clean, explainable matches in milliseconds.

In this notebook you will:

1. Install the SDK
2. Build your first search index from a Pandas DataFrame
3. Run a fuzzy match query
4. Understand the explainable output M|BOX returns

## 1. Install the SDK

M|BOX is distributed on PyPI. Run the cell below to install it (or run this in your terminal without the `!`).

In [1]:
!pip install mbox pandas


[notice] A new release of pip is available: 25.3 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Create some sample data

Real projects will load data from a CSV, database, or warehouse table (see the next notebook, `02_loading_your_own_csv.ipynb`). For now, we'll use a small in-memory DataFrame so you can see the whole workflow end to end.

Notice the data isn't clean, that's intentional. M|BOX is designed to handle exactly this kind of messiness.

In [2]:
import pandas as pd
from mbox.indexing import TableIndexer

# A small, slightly messy customer table
df = pd.DataFrame({
    "full_name": ["Robert Smith", "Katharina Müller", "Jochen Meyer"],
    "city": ["Konstanz", "Berlin", "Munich"]
})

df

mbpie: 33 modules, 566 methods, 8 classes, 18 enums
  args: 426 required, 254 optional, 37 keywords, 39 flags, 26 arrays
  types: 372 int, 337 str, 1 double, 72 object


,full_name,city
0,Robert Smith,Konstanz
1,Katharina Müller,Berlin
2,Jochen Meyer,Munich


## 3. Build your first index

M|BOX works in two phases:

- **Indexing phase**: your DataFrame is compiled into an optimized, in-memory search structure.
- **Recall phase**: you query that structure with fuzzy, explainable matching.

Let's build the index. We tell `TableIndexer` which columns should be searchable via `index_columns`.

In [6]:
index = TableIndexer.create_index(df, index_columns=["full_name", "city"], tmp_dir = "tmp_index")

# Confirm the index was built successfully
index

After creating your index, the index files can be found under `/tmp_index` directory.

## Peeking at the index with `head()`

Before you start querying, it can be useful to confirm the index actually holds what you expect. `TableIndex.head()` retrieves the first `n` records directly from the underlying compiled base, a quick sanity check that your data made it in correctly.

In [ ]:
index.head(n=5)

This returns the first `n` records straight from the compiled index, not the original DataFrame, so it's a good way to confirm the data was ingested as expected, including any columns you didn't explicitly mark as searchable. Since our sample data only has 3 rows, `head(n=5)` simply returns all of them.

## 4. Run your first match

Now for the fun part. Let's search for `"Bob Smit"`, a nickname *and* a typo, in `"Konstanz"`.

A naive exact-match lookup would fail here. M|BOX won't.

In [7]:
results = index.match(
    full_name="Bob Smit",
    city="Konstanz",
    include_field_scores=True
)

results

,query_row,index_row,full_name_candidate,city_candidate,overall_score,full_name_score,city_score
0,0,0,Robert Smith,Konstanz,78,56,100


## 5. Understanding the output

Your results DataFrame includes:

| Column | Meaning |
|---|---|
| `query_row` | Which input query this row corresponds to (0-indexed) |
| `index_row` | The row in your original DataFrame that matched |
| `overall_score` | Overall match confidence, 0–100 |
| `full_name_candidate` | The actual indexed value that matched |
| `full_name_score` | How well `full_name` specifically matched, 0–100 |
| `city_candidate` / `city_score` | Same, for the `city` field |

Even though we searched for `"Bob Smit"`, M|BOX correctly surfaced **"Robert Smith"** as the top candidate, resolving both the nickname (*Bob → Robert*) and the typo (*Smit → Smith*), and gave us a transparent, field-by-field score explaining *why*.

This explainability is the core of what makes M|BOX different from a black-box embedding search: every match can be inspected, validated, and trusted.

## 6. Matching multiple records at once

So far we've searched for one record at a time. In practice, you'll usually have a whole batch of records to resolve, a CSV of leads, a list of incoming support tickets, a nightly dedup job.

M|BOX handles this naturally: instead of passing a single string to each field, pass a **list**. Each position in the list is treated as its own query row, matched independently against the index.

In [15]:
results = index.match(
    full_name=["Bob Smit", "Katarina Muller"],
    city=["Konstanz", "Berlin"],
    include_field_scores=True,
    include_queries=True
)

results

,query_row,full_name_query,city_query,index_row,full_name_candidate,city_candidate,overall_score,full_name_score,city_score
0,0,Bob Smit,Konstanz,0,Robert Smith,Konstanz,78,56,100
1,1,Katarina Muller,Berlin,1,Katharina Müller,Berlin,93,86,100


### Reading a batch result

With multiple queries, results stack one row per query. A few columns are new:

| Column | Meaning |
|---|---|
| `query_row` | Which query this row belongs to, `0` = first pair, `1` = second |
| `full_name_query` / `city_query` | The raw input you searched for (shown via `include_queries=True`) |
| `full_name_score` / `city_score` | Field-level confidence, *why* the overall score landed where it did |

Notice **row 0** scores lower (`56`) than **row 1** (`86`) on `full_name_score`: *"Bob Smit" → "Robert Smith"* resolves both a nickname *and* a typo, while *"Katarina Muller" → "Katharina Müller"* is just a spelling/accent difference, a smaller fuzziness gap. `city_score` is `100` for both, since the cities were typed correctly.

Same explainability as before, it just scales across a batch.

## Next steps

You've built an index and run your first explainable match. From here:

- **`02_loading_your_own_csv.ipynb`**, swap this demo data for your own CSV file
- **`03_saving_and_loading_indexes.ipynb`**, save your compiled index to disk so you don't rebuild it on every run
- **`2-data-harmonization/`**, handle accents, umlauts, and nicknames automatically with `CharacterMapping` and `AliasSet`
- **`3-recall-tuning/`**, control exactly how matches are scored with field weights and match modes

*M|BOX is currently in `beta`. Breaking changes may occur in minor releases until version `1.0.0`.*